In [0]:
from pyspark.sql.functions import *

In [0]:
source_path = "/Volumes/retailnova/bronze/retailnova_source/retailnova_datasets/stores/"
silver_table = "retailnova.silver.stores"

In [0]:
# Read source files
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(source_path)


In [0]:
print("Source records:", df.count())

# Remove invalid store IDs
df = df.filter(
    col("store_id").isNotNull()
)

Source records: 500


In [0]:
# Trim string columns
df = df.withColumn("store_id", trim(col("store_id")))
df = df.withColumn("store_name", trim(col("store_name")))
df = df.withColumn("city", trim(col("city")))
df = df.withColumn("state", trim(col("state")))
df = df.withColumn("region", trim(col("region")))
df = df.withColumn("store_type", trim(col("store_type")))

In [0]:
# Standardize text
df = df.withColumn("city", initcap(lower(col("city"))))
df = df.withColumn("state", initcap(lower(col("state"))))
df = df.withColumn("region", initcap(lower(col("region"))))
df = df.withColumn("store_type", initcap(lower(col("store_type"))))

In [0]:
# Remove duplicate stores
df = df.dropDuplicates(["store_id"])

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print("Stores Silver created successfully")

Stores Silver created successfully


In [0]:
spark.sql("""
    SELECT COUNT(*)
    FROM retailnova.silver.stores
""").show()

+--------+
|COUNT(*)|
+--------+
|     500|
+--------+

